In [ ]:
import pandas as pd
from db import engine, admin_engine
from style import setup_style
from stop_words import STOP_WORDS
from news_helpers import (
    compute_embeddings, load_embeddings, compute_umap, plot_by_source,
    run_bertopic, save_topics, show_topics, plot_umap_by_topic, plot_topic_timeline,
)
setup_style()

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
BATCH_SIZE = 256
RUN_ID     = "run_01"
NEWS_LANG  = "all"

total    = pd.read_sql("SELECT count(*) AS n FROM news_events", engine).iloc[0]["n"]
with_emb = pd.read_sql("SELECT count(*) AS n FROM news_events WHERE embedding IS NOT NULL", engine).iloc[0]["n"]
print(f"Artigos: {total:,}  ·  com embedding: {with_emb:,}")

In [ ]:
compute_embeddings(engine, admin_engine, MODEL_NAME, BATCH_SIZE)

In [ ]:
df_emb, X = load_embeddings(engine)

In [ ]:
df_emb = compute_umap(df_emb, X)
plot_by_source(df_emb)

In [ ]:
df_emb, topic_model = run_bertopic(df_emb, X, STOP_WORDS)

In [ ]:
import json as _json
from sqlalchemy import text as _text

_run_id     = RUN_ID
_run_config = {'embedding_model': MODEL_NAME, 'news_lang': NEWS_LANG,
               'stop_words': str(len(STOP_WORDS)) + "_words", 'corpus_size': len(df_emb)}
_topic_info = topic_model.get_topic_info()

with admin_engine.begin() as conn:
    for _, row in _topic_info.iterrows():
        tid        = int(row.Topic)
        words      = topic_model.get_topic(tid)
        words_json = _json.dumps([{'word': w, 'score': round(s, 4)} for w, s in (words or [])])
        label      = str(row.get('Name', row.get('Representation', '')))
        conn.execute(_text("""
            INSERT INTO news_topic_definitions (run_id, topic_id, label, top_words, article_count, config)
            VALUES (:run, :tid, :label, cast(:words AS jsonb), :cnt, cast(:cfg AS jsonb))
            ON CONFLICT (run_id, topic_id) DO UPDATE SET
                label         = EXCLUDED.label,
                top_words     = EXCLUDED.top_words,
                article_count = EXCLUDED.article_count,
                config        = EXCLUDED.config
        """), {'run': _run_id, 'tid': tid, 'label': label,
               'words': words_json, 'cnt': int(row.Count),
               'cfg': _json.dumps(_run_config)})

    for _, row in df_emb.iterrows():
        conn.execute(_text("""
            UPDATE news_events SET topic_id = :tid, topic_run = :run WHERE id = :id
        """), {'tid': int(row.topic_id), 'run': _run_id, 'id': int(row.id)})

print(f'Tópicos salvos — run_id={_run_id}  ·  {len(_topic_info)} tópicos  ·  {len(df_emb):,} artigos tagueados')

In [ ]:
show_topics(engine)
plot_umap_by_topic(df_emb)
plot_topic_timeline(df_emb)